# Earth System Grid Federation Data Access

The simplest way to search datasets programmatically within the Earth System Grid Federation (ESGF) is to use [`intake-esgf`](https://intake-esgf.readthedocs.io/). It works like `intake-esm`, but instead of loading a static catalog, `intake-esgf` populates its catalog by querying ESGF index nodes. There is also a new graphical search interface called [Metagrid](https://aims2.llnl.gov/search), but it does not yet federate data from all ESGF nodes.

ESGF provides two different mechanisms to retrieve data, the legacy `Solr` index and the new `Globus` index. ESGF is decommissioning the old indexing service based on SOLR, so methods relying on it might break. `intake-esgf` works with both, so should allow for a graceful transition.

What `intake-esgf` does is send search queries to a list of index nodes and aggregates the results. You can configure the list of index nodes it queries, especially since some of the nodes are very slow to respond. For this notebook, we'll query only Globus nodes to speed things up, but to ensure you find all the available data, call `intake_esgf.conf.set(all_indices=True)` to search all nodes.

In [1]:
# NBVAL_IGNORE_OUTPUT

import intake_esgf

# Silence warnings from STAC publishing error
intake_esgf.conf.set(
    indices={"discovery.east.esgf.io": False, "discovery.west.esgf.io": False},
    # local_cache="/notebook_dir/writable-workspace/cache"
)

# Show the configuration
print(intake_esgf.conf)

# Initialize an empty ESGF catalog
cat = intake_esgf.ESGFCatalog()

# Launch a search query.
# Here we're looking for any variable related to humidity within the CMIP6 SSP2-4.5 experiment.
# Results will be stored in a dictionary with keys defined by the `facets` argument.
cat.search(
    project="CMIP6",
    variable_id=["hurs"],
    table_id="Amon",
    experiment_id=["ssp126", "ssp245", "ssp370", "ssp585"],
)

Output()

additional_df_cols: []
break_on_error: true
confirm_download: false
default_open_kwargs:
  chunks: auto
  compat: override
  coords: minimal
  data_vars: minimal
default_project: CMIP7
download_db: ~/.config/intake-esgf/download.db
esg_dataroot:
- /p/css03/esgf_publish
- /eagle/projects/ESGF2/esg_dataroot
- /global/cfs/projectdirs/m3522/cmip6/
- /glade/campaign/collections/cmip.mirror
globus_indices:
  ESGF2-US-1.5-Catalog: true
local_cache:
- ~/.esgf/
logfile: ~/.config/intake-esgf/esgf.log
num_threads: 6
print_log_on_error: false
requests_cache:
  cache_name: intake-esgf/requests-cache.sqlite
  expire_after: 3600
  use_cache_dir: true
slow_download_threshold: 0.5
solr_indices:
  esg-dn1.nsc.liu.se: false
  esgf-data.dkrz.de: false
  esgf-node.ipsl.upmc.fr: false
  esgf.ceda.ac.uk: false
  esgf.nci.org.au: false
stac_indices:
  discovery.east.esgf.io: false
  discovery.west.esgf.io: false



Summary information for 1699 results:
mip_era                                                     [CMIP6]
activity_id                                           [ScenarioMIP]
institution_id    [AWI, CAS, CCCR-IITM, CCCma, CMCC, CNRM-CERFAC...
source_id         [AWI-CM-1-1-MR, CAS-ESM2-0, FGOALS-f3-L, FGOAL...
experiment_id                      [ssp126, ssp245, ssp370, ssp585]
member_id         [r1i1p1f1, r2i1p1f1, r3i1p1f1, r4i1p1f1, r5i1p...
table_id                                                     [Amon]
variable_id                                                  [hurs]
grid_label                                            [gn, gr, gr1]
dtype: object

You can get a sense of what datasets are available with the `model_group` method, which counts the number of unique combinations of `source_id, member_id` and `grid_label`.

Other useful methods are `remove_ensembles`, which picks only one `member_id` (the smallest 4 integer values for the *ripf* code, and `remove_incomplete`, which filters model groups according to criteria you can define. See the [docs](https://intake-esgf.readthedocs.io/en/latest/modelgroups.html) for details.

In [2]:
# NBVAL_IGNORE_OUTPUT

# Keep only one member per model, experiment and grid.
cat.remove_ensembles()

# Remove models groups that don't have the four SSPs.
cat.remove_incomplete(lambda df: len(df) == 4)

cat.model_groups()

source_id         member_id  grid_label
ACCESS-CM2        r1i1p1f1   gn            4
ACCESS-ESM1-5     r1i1p1f1   gn            4
AWI-CM-1-1-MR     r1i1p1f1   gn            4
CanESM5           r1i1p1f1   gn            4
CanESM5-1         r1i1p1f1   gn            4
CanESM5-CanOE     r1i1p2f1   gn            4
CAS-ESM2-0        r1i1p1f1   gn            4
CESM2-WACCM       r1i1p1f1   gn            4
CMCC-CM2-SR5      r1i1p1f1   gn            4
CMCC-ESM2         r1i1p1f1   gn            4
CNRM-CM6-1        r1i1p1f2   gr            4
CNRM-CM6-1-HR     r1i1p1f2   gr            4
CNRM-ESM2-1       r1i1p1f2   gr            4
EC-Earth3         r1i1p1f1   gr            4
EC-Earth3-Veg     r1i1p1f1   gr            4
EC-Earth3-Veg-LR  r1i1p1f1   gr            4
FGOALS-f3-L       r1i1p1f1   gr            4
FGOALS-g3         r1i1p1f1   gn            4
GFDL-ESM4         r1i1p1f1   gr1           4
GISS-E2-1-G       r1i1p1f2   gn            4
GISS-E2-1-H       r1i1p1f2   gn            4
GISS-E2-2-G    

Now we'll try to access some data. For small queries, a good approach is to use streaming, rather than downloading the whole thing. Here we'll just ask for simulations from CanESM5, and try to stream some data. Getting the file information can take some time.

In [3]:
# NBVAL_IGNORE_OUTPUT

# Let's focus the search on one single model to speed up the rest of the notebook
cat.search(
    project="CMIP6",
    source_id="CanESM5",
    variable_id=["hurs"],
    table_id="Amon",
    experiment_id=["ssp126", "ssp245", "ssp370", "ssp585"],
)
cat.remove_ensembles()

# The `prefer_streaming` argument specifies that we'd rather not download entire files.
# When True, the `add_measures` argument triggers search for variables that are referenced
# in the `cell_measure` attribute, such as `areacella` or `areacello`.
dsd = cat.to_dataset_dict(
    prefer_streaming=True,
    add_measures=False,
    open_kwargs={"chunks": {"lat": 50, "lon": 50, "time": -1}},
)

Output()

Output()

/opt/conda/envs/birdy/lib/python3.12/site-packages/intake_esgf/catalog.py:745: FutureWarning: In a future version of xarray the default value for data_vars will change from data_vars='all' to data_vars=None. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set data_vars explicitly.
  ds[key] = xr.open_mfdataset(sorted(files), **open_kwargs)
/opt/conda/envs/birdy/lib/python3.12/site-packages/intake_esgf/catalog.py:745: FutureWarning: In a future version of xarray the default value for data_vars will change from data_vars='all' to data_vars=None. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set data_vars explicitly.
  ds[k

In [4]:
# NBVAL_IGNORE_OUTPUT

# Here the result is keyed by experiment_id.
dsd["ssp370"]["hurs"]

<xarray.DataArray 'hurs' (time: 1032, lat: 64, lon: 128)> Size: 34MB
dask.array<open_dataset-hurs, shape=(1032, 64, 128), dtype=float32, chunksize=(1032, 50, 50), chunktype=numpy.ndarray>
Coordinates:
  * time     (time) object 8kB 2015-01-16 12:00:00 ... 2100-12-16 12:00:00
  * lat      (lat) float64 512B -87.86 -85.1 -82.31 -79.53 ... 82.31 85.1 87.86
  * lon      (lon) float64 1kB 0.0 2.812 5.625 8.438 ... 348.8 351.6 354.4 357.2
    height   float64 8B ...
Attributes:
    standard_name:  relative_humidity
    long_name:      Near-Surface Relative Humidity
    comment:        SRH*100, ScenarioMIP_table_comment: The relative humidity...
    units:          %
    original_name:  SRH
    history:        mltby100 2019-04-30T18:46:33Z altered by CMOR: Treated sc...
    cell_methods:   area: time: mean
    cell_measures:  area: areacella
    _ChunkSizes:    [  1  64 128]

By default, the ``to_dataset_dict`` method downloads files locally. If you already hold datasets locally, you can specify the ``esg_dataroot`` in the configuration. You can also specify the local_cache where missing datasets will be downloaded.

Please check the [documentation](https://intake-esgf.readthedocs.io/) for more details on how to use `to_dataset_dict`.